In [ ]:
#only for cloab
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#formatting of certain escape seq
with open('/content/drive/MyDrive/Gpt /dataset.txt','r',encoding='utf-8-sig') as f:
  textData=f.read()

In [ ]:
textData

'The Project Gutenberg eBook of The Harvard Classics Volume 38\n    \nThis eBook is for the use of anyone anywhere in the United States and\nmost other parts of the world at no cost and with almost no restrictions\nwhatsoever. You may copy it, give it away or re-use it under the terms\nof the Project Gutenberg License included with this eBook or online\nat www.gutenberg.org. If you are not located in the United States,\nyou will have to check the laws of the country where you are located\nbefore using this eBook.\n\nTitle: The Harvard Classics Volume 38\n\nAuthor: Various\n\n\n        \nRelease date: May 1, 2004 [eBook #5694]\n                Most recently updated: December 29, 2020\n\nLanguage: English\n\nOther information and formats: www.gutenberg.org/ebooks/5694\n\nCredits: Produced by David Turner, Charles Franks and the Online Distributed Proofreading Team\n\n\n*** START OF THE PROJECT GUTENBERG EBOOK THE HARVARD CLASSICS VOLUME 38 ***\n\n\n\n\nProduced by David Turner, Charles F

In [ ]:
#removing the unnecessary part and keeping only the main content
start_text='*** START OF THE PROJECT GUTENBERG EBOOK'
end_text='*** END OF THE PROJECT GUTENBERG EBOOK'
start_ind=textData.find(start_text)
end_ind=textData.find(end_text)

if start_ind!=-1 and end_ind!=-1:
  start=textData.find('\n',start_ind)+1
  textData=textData[start:end_ind].strip()

print(textData)

Produced by David Turner, Charles Franks
and the Online Distributed Proofreading Team.




The Harvard Classics Volume 38
Scientific Papers (Physiology, Medicine, Surgery, Geology)




CONTENTS

THE OATH OF HIPPOCRATES


THE LAW OF HIPPOCRATES

JOURNEYS IN DIVERSE PLACES ... AMBROISE PARE
TRANSLATED BY STEPHEN PAGET

ON THE MOTION OF THE HEART AND BLOOD IN ANIMALS
WILLIAM HARVEY. . . TRANSLATED BY ROBERT WILLIS

THE THREE ORIGINAL PUBLICATIONS ON VACCINATION
AGAINST SMALLPOX . ... .. EDWARD JENNER

THE CONTAGIOUSNESS OF PUERPERAL FEVER
O. W. HOLMES

ON THE ANTISEPTIC PRINCIPLE OF THE PRACTICE OF SURGERY
LORD LISTER

THE PHYSIOLOGICAL THEORY OF FERMENTATION
LOUIS PASTEUR
TRANSLATED BY F. FAULKNER AND D. C. ROBB (Revised)

THE GERM THEORY AND ITS APPLICATIONS TO MEDICINE AND
SURGERY (Revised) . ... .. LOUIS PASTEUR
TRANSLATED BY H. C. ERNST

ON THE EXTENSION OF THE GERM THEORY TO THE ETIOLOGY
OF CERTAIN COMMON DISEASES (Revised) LOUIS PASTEUR
TRANSLATED BY H. C. ERNST

PREJUDICES WHICH H

In [ ]:
print("Length",len(textData))
print(textData[:100])

Length 928454
Produced by David Turner, Charles Franks
and the Online Distributed Proofreading Team.




The Harva


In [ ]:
chars=sorted(list(set(textData)))
vocab=len(chars)
print(' '.join(chars))
print(vocab)


   ! " & ' ( ) * , - . / 0 1 2 3 4 5 6 7 8 9 : ; = ? A B C D E F G H I J K L M N O P Q R S T U V W X Y Z [ ] a b c d e f g h i j k l m n o p q r s t u v w x y z
81


In [ ]:
!pip install tokenizers

In [ ]:
#creating custom tokenizer
#find 5000 most freq pair with atleat 2 occ and a special token
from tokenizers import ByteLevelBPETokenizer
tokenizer=ByteLevelBPETokenizer()
tokenizer.train_from_iterator([textData],vocab_size=5000,min_frequency=2,special_tokens=["<|endoftext|>"])
vocab_size=tokenizer.get_vocab_size()

In [ ]:
def encode(s):
  endStr=tokenizer.encode(s)
  return endStr.ids

def decode(l):
  return tokenizer.decode(l)

In [ ]:
enc=encode('Hello')
print(enc)
print(decode(enc))
print(vocab_size)

[40, 794, 79]
Hello
5000


In [ ]:
#making a tensor of entire dataset
import torch
data=torch.tensor(encode(textData),dtype=torch.long)
print(data.shape,data.dtype)
print(data[:200])

torch.Size([242637]) torch.int64
tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12, 1412,  282,
         630,  525, 3666,  692,  199,  410,  260, 1066,   76,  528,  568,  485,
         888, 1735,  534,  352,  492,  814,  293,  349,   69,  354,   14, 1231,
         199,  592,  538,  282,   86,  507, 4162,  526, 2555, 1088,  391, 1182,
        1377,   24,  199,   51, 4868,  534,   65, 2252,  594,   48,   72, 1455,
        3583,   12, 4828,   12, 2814, 3302,   12, 4781,   89,    9, 1231,  199,
        3646,   52, 1781,   51,  199,  199, 1476,  535, 1374,   40,  907,  538,
          41,   48, 2533,   35,   50, 1374, 1234,  563,  199, 1476,  624,   33,
          55,  907,  538,   41,   48, 2533,   35,   50, 1374, 1234,  199,  199,
          42, 3088,   51, 1448,  568, 2881,  697, 1049,  534,   44, 2202, 1234,
         869,  399,   45,   34, 1600, 3648,  534, 1173,   37,  199,   52, 4630,
        2501,  456,   52,   37,   48,  635,   46, 2443,   39,   37,   52,  199,
       

In [ ]:
#splitting into train and test (85/15)
n=int(len(data)*0.85)
train_data=data[:n]
val_data=data[n:]

In [ ]:
block_size=32
x=train_data[:block_size]
y=train_data[1:block_size+1]
for t in range(block_size):
  context=x[:t+1]
  val=y[t]
  print(f"context:{context} target:{val}")

context:tensor([48]) target:3835
context:tensor([  48, 3835]) target:523
context:tensor([  48, 3835,  523]) target:357
context:tensor([  48, 3835,  523,  357]) target:568
context:tensor([  48, 3835,  523,  357,  568]) target:881
context:tensor([  48, 3835,  523,  357,  568,  881]) target:325
context:tensor([  48, 3835,  523,  357,  568,  881,  325]) target:4712
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712]) target:800
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800]) target:12
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12]) target:1412
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12, 1412]) target:282
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12, 1412,  282]) target:630
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12, 1412,  282,
         630]) target:525
context:tensor([  48, 3835,  523,  357,  568,  881,  325, 4712,  800,   12, 1412, 

In [ ]:
torch.manual_seed(1330)
batch_size=32
block_size=32

#pick random start ind and extract bloack size chunks and stack to make a 2d tensor
def get_batch(split):
  data= train_data if split=='train' else val_data
  randStartInd=torch.randint(0,len(data)-block_size,(batch_size,)) #pyth expects tuple here so comma
  x=torch.stack([data[i:i+block_size] for i in randStartInd])
  y=torch.stack([data[i+1:i+block_size+1] for i in randStartInd])
  return x,y

x,y=get_batch('train')
print(x.shape,y.shape)
print(x)
print(y)

torch.Size([32, 32]) torch.Size([32, 32])
tensor([[ 364,  260,  538,  ...,  427,  336,   80],
        [1351, 1641,  870,  ...,  270, 2037, 3813],
        [ 594,   70,  444,  ...,  429,  260, 1230],
        ...,
        [ 199,  533,  288,  ...,  509, 1819,   14],
        [2417, 1235,  286,  ...,  808, 1246,  270],
        [ 890,  353,  260,  ...,  399,  384,  448]])
tensor([[ 260,  538,  305,  ...,  336,   80, 4079],
        [1641,  870,  515,  ..., 2037, 3813, 1252],
        [  70,  444,   14,  ...,  260, 1230, 1792],
        ...,
        [ 533,  288,  300,  ..., 1819,   14,  390],
        [1235,  286,  278,  ..., 1246,  270,  540],
        [ 353,  260,  199,  ...,  384,  448, 4173]])


In [ ]:
#how we will use our data for training the transformer
for i in range(batch_size):
  for j in range(block_size):
    context=x[i,:j+1]
    pred=y[i][j]
    print(f"context:{context} target:{pred}")

context:tensor([364]) target:260
context:tensor([364, 260]) target:538
context:tensor([364, 260, 538]) target:305
context:tensor([364, 260, 538, 305]) target:735
context:tensor([364, 260, 538, 305, 735]) target:568
context:tensor([364, 260, 538, 305, 735, 568]) target:486
context:tensor([364, 260, 538, 305, 735, 568, 486]) target:85
context:tensor([364, 260, 538, 305, 735, 568, 486,  85]) target:199
context:tensor([364, 260, 538, 305, 735, 568, 486,  85, 199]) target:261
context:tensor([364, 260, 538, 305, 735, 568, 486,  85, 199, 261]) target:1993
context:tensor([ 364,  260,  538,  305,  735,  568,  486,   85,  199,  261, 1993]) target:12
context:tensor([ 364,  260,  538,  305,  735,  568,  486,   85,  199,  261, 1993,   12]) target:329
context:tensor([ 364,  260,  538,  305,  735,  568,  486,   85,  199,  261, 1993,   12,
         329]) target:260
context:tensor([ 364,  260,  538,  305,  735,  568,  486,   85,  199,  261, 1993,   12,
         329,  260]) target:1272
context:tensor([ 

#A Simple Biagram Language Model

In [ ]:
# a simle biagram model that has no memory and only uses current word for pred
#it uses a simple lookup table for pred
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1330)

class BigramLanguageModel(nn.Module):
  def __init__(self,vocab_size):
    super().__init__()
    #table creation
    self.token_embedding_table=nn.Embedding(vocab_size,vocab_size)

  def forward(self,idx,target=None):
    #idx=b*t(batch,time)
    logits=self.token_embedding_table(idx)
    if target==None:
      loss=None
    else:
      B,T,C=logits.shape #logits will add channel(here vocab size)
      logits=logits.view(B*T,C)# as cross entropy has a specific tensor syntax
      target=target.view(B*T)
      loss=F.cross_entropy(logits,target)

    return logits,loss

  def generate(self,idx,max_generate):
    for i in range(max_generate):
      logits,loss=self.forward(idx)
      #pred is in last one so focus on that
      logits=logits[:,-1,:] #need last as it is our whole text(rest are in bw train sample)
      prob=F.softmax(logits,dim=-1) #apply softmax on the last dim not on the batch(b,c)
      idx_next=torch.multinomial(prob,num_samples=1)#take one sample from it (b,1)
      idx=torch.cat((idx,idx_next),dim=1);

    return idx

model=BigramLanguageModel(vocab_size)
logits,loss=model.forward(x,y) #from pre
print(logits.shape)
print(loss)

torch.Size([1024, 5000])
tensor(8.9752, grad_fn=<NllLossBackward0>)


In [ ]:
#starting generation from our special token as it will act as end and start both
import torch
start_token_id=tokenizer.token_to_id("<|endoftext|>")
idx=torch.tensor([[start_token_id]],dtype=torch.long)
print(idx)
generate=model.generate(idx,max_generate=100)
print(generate)
print(decode(generate[0].tolist()))

tensor([[0]])
tensor([[   0, 4967, 1540, 1713,  182, 4097,  359,  118, 3390, 2145, 1543, 3774,
         3330,  250, 3079, 3445, 1878, 2548, 2265, 1697, 1783, 3146,  683, 1301,
         3338, 4452,   34, 2971, 4656,  957, 3367, 3373, 1961,    5, 2824, 3693,
         3863, 4211,  273, 4986, 3722, 2901, 2617, 1072, 2728, 1450,  766, 2206,
         2098, 4711,  223, 2768, 1033, 3645,  955, 4847, 4982, 3635, 3680, 3595,
         3400, 1969, 4520,  154, 3102, 2436, 1128,  511, 3510, 4125, 4725, 3820,
         4729, 2394, 4232, 3036, 4225, 4480, 2826, 2133, 1222,  713, 2898, 3400,
         1398, 2003,  849, 4700, 2859, 2339, 4967, 1170,  802, 4943, 1293, 3867,
         3142, 4621,  550, 1100, 4643]])
cers human too� creatain� reportISnedORM function� raisedstrong sick age distrib near grammes See onlyhave embryieveB keepjudless fa ma remained% Figstate hither regularndnce ransomiseseav deg preserampxygenbutolethousand�gy fourAftertr secondarymain precautions pal returning stated volume compre

In [ ]:
#training using adam optimiser
optimiser=torch.optim.AdamW(model.parameters(),lr=1e-3)
batch_size=32
training_step=10000

for step in range(training_step):
  xb,yb=get_batch('train')
  logits,loss=model.forward(xb,yb) #calc logits and loss
  optimiser.zero_grad(set_to_none=True) #set prev grad to 0 to remove inf
  loss.backward() #back prop
  optimiser.step() #update weights

  if step%400==0:
    print(f'step:{step} loss:{loss.item()}')

print('--------')
print(f'final Loss:{loss.item()}')

step:0 loss:9.066106796264648
step:400 loss:8.702919960021973
step:800 loss:8.415155410766602
step:1200 loss:8.053214073181152
step:1600 loss:7.873037338256836
step:2000 loss:7.63845157623291
step:2400 loss:7.372746467590332
step:2800 loss:6.9725470542907715
step:3200 loss:6.813873291015625
step:3600 loss:6.585742950439453
step:4000 loss:6.421133995056152
step:4400 loss:6.316802024841309
step:4800 loss:6.112058162689209
step:5200 loss:5.9825029373168945
step:5600 loss:5.864264488220215
step:6000 loss:5.735032081604004
step:6400 loss:5.541348457336426
step:6800 loss:5.420754432678223
step:7200 loss:5.3731255531311035
step:7600 loss:5.275778293609619
step:8000 loss:5.31451416015625
step:8400 loss:5.096238136291504
step:8800 loss:5.077060699462891
step:9200 loss:5.069199085235596
step:9600 loss:4.950926780700684
--------
final Loss:4.999682426452637


In [ ]:
for step in range(2000):
  xb,yb=get_batch('train')
  logits,loss=model.forward(xb,yb) #calc logits and loss
  optimiser.zero_grad(set_to_none=True) #set prev grad to 0 to remove inf
  loss.backward() #back prop
  optimiser.step() #update weights

  if step%400==0:
    print(f'step:{step} loss:{loss.item()}')

print('--------')
print(f'final Loss:{loss.item()}')

step:0 loss:4.861748695373535
step:400 loss:4.704867362976074
step:800 loss:4.786357402801514
step:1200 loss:4.683818340301514
step:1600 loss:4.733726978302002
--------
final Loss:4.722290515899658


In [ ]:
#starting generation from our special token as it will act as end and start both
generate=model.generate(idx,max_generate=100)
print(generate)
print(decode(generate[0].tolist()))

tensor([[   0, 2816, 1171, 2029, 3297, 4727, 4869, 1516,  594, 3692, 2818, 2741,
          560,  270,  199, 2291, 1258, 3140, 1588, 2004, 4696, 3493, 1485, 4968,
         2864, 3361, 2972,   66, 2184, 1574, 4933, 1047,  657, 3531, 2358, 4512,
         2418, 3938, 4269,  516, 1276, 4447, 4094, 2157, 1585, 2296, 2660, 2845,
         2285, 3262, 3503, 4939, 1258, 3821, 2106, 3634, 4008, 2424, 2387, 3563,
         2787, 3207,  779,  336, 1906, 4321, 2795, 1853, 1919, 4231, 4612, 3424,
          633,  159,  919,   71,  322,  327,  690,  512,  260, 1495,  429,  315,
          300,  948, 2406, 1695,  517,  378, 1778,  554, 2252, 2865,  523, 1064,
         1923, 2969, 4741, 3187,  220]])
ivesselves need carbolicumm ulcerationude (althoughague physicians form of
bus showhers progressThisicacy BLOODidencecav globeuce cultivb bringestion exclusstancery love sideshireatompret sufficiently . cent Suchisioninousirmsame consist certainly establ secremod CIRCUL showbor thingdecomposition recorded mal 

# Mathmatical trick in self attention

In [ ]:
#we can take take running sum by using for loop but using the lower tri help be doing matrix mul
T = 3
tril = torch.tril(torch.ones(T, T))
tril

tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

In [ ]:
import torch

T = 3
x = torch.tensor([[10.],
                  [20.],
                  [30.]])

wei_sum = torch.tril(torch.ones(T, T))
result_sum = wei_sum @ x
print(result_sum)

tensor([[10.],
        [30.],
        [60.]])


In [ ]:
#for making avg of the running sum use softmax
import torch.nn.functional as F
wei_avg = torch.zeros((T, T))
wei_avg = wei_avg.masked_fill(wei_sum == 0, float('-inf'))
wei_avg = F.softmax(wei_avg, dim=-1)
result_avg = wei_avg @ x
print(wei_avg)
print(result_avg)

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
tensor([[10.0000],
        [15.0000],
        [20.0000]])


# The attention introduced in the 2017 paper
#$$Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
#self attention as described in the Attention is all you need paper
#pytorch uses last two dim for mul(or dot)
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1330)

B,T,C=4,8,32
head_size=16 #for attention only allow 16 channel interaction(for easier math)

x=torch.randn(B,T,C)
key=nn.Linear(C,head_size,bias=False) #general dense layr with in_channel=c,out=head_size and dont add bias
query=nn.Linear(C,head_size,bias=False)
value=nn.Linear(C,head_size,bias=False)

k=key(x) #what i contain
q=query(x) #what i am looking at
v=value(x) #if you take this this will be the res
wei=q @ k.transpose(-2,-1)  #(b,t,16).(b,t,16) so t,16.t,16 cant so transpose the last two diim-->t,16.16,t
#print(wei) #wei->b,t,t , HOW MUCH FOCUS TO GIVE FOR THAT TOKEN
print(wei.var())
#MASKING SO IT DONT COMM WITH FUTURE TOKENS
tril=torch.tril(torch.ones(T,T)) #lower triang matrix
wei=wei.masked_fill(tril==0,float('-inf')) #sort of decoder block(in encoder generally remove)
wei=F.softmax(wei,dim=-1)

#final with attention and impact
out=wei@v
print(out)

tensor(1.0351, grad_fn=<VarBackward0>)
tensor([[[-2.9944e-01,  1.0922e+00,  5.1968e-01,  1.0971e-01,  8.8391e-01,
          -3.9054e-01, -1.0478e-01,  1.0455e-01,  5.0288e-01, -6.6690e-01,
          -1.4475e+00, -4.1909e-01, -3.5139e-01, -3.8202e-02, -4.9438e-01,
          -9.8271e-01],
         [ 1.7783e-02, -2.0074e-01, -3.9205e-01,  4.2562e-01,  1.1619e-01,
           1.6593e-01,  1.4157e-01,  3.7790e-01,  7.7090e-01,  2.1190e-01,
           4.0143e-01, -1.8973e-02, -5.9135e-01, -3.0499e-01, -6.4223e-01,
           5.0080e-01],
         [ 9.0964e-02,  6.4891e-02, -1.6988e-01,  1.7826e-01,  4.0699e-01,
           9.6323e-02, -1.7516e-02,  9.4030e-02,  4.5045e-01, -2.0349e-02,
          -1.2953e-01, -5.6250e-03, -3.8142e-01, -1.2838e-01, -4.0509e-01,
          -5.5885e-02],
         [ 1.1590e-01, -1.3129e-01, -2.8444e-01,  1.4438e-01,  2.9375e-01,
           1.3484e-01,  1.0157e-01,  2.9901e-01,  4.6330e-01,  1.0078e-01,
           2.9398e-02,  8.1435e-02, -4.3474e-01, -7.3065e-02, -3

In [ ]:
#why the dk term is important
#softmax has a tendency to shift towards the larger numbers
#as we are doing dot valus will scale so o prevent that we do the scaling(exploding variance)
#it also keeps the variance in check(as grows exactly by head size)

k=key(x)
q=query(x)
v=value(x)
wei=q @ k.transpose(-2,-1) * head_size**-0.5  #TO PREVENT EXPLODING VAR
print(wei.var())

tril=torch.tril(torch.ones(T,T)) #lower triang matrix
wei=wei.masked_fill(tril==0,float('-inf')) #sort of decoder block(in encoder generally remove)
wei=F.softmax(wei,dim=-1)

#final with attention and impact
out=wei@v
print(out)

tensor(0.0647, grad_fn=<VarBackward0>)
tensor([[[-2.9944e-01,  1.0922e+00,  5.1968e-01,  1.0971e-01,  8.8391e-01,
          -3.9054e-01, -1.0478e-01,  1.0455e-01,  5.0288e-01, -6.6690e-01,
          -1.4475e+00, -4.1909e-01, -3.5139e-01, -3.8202e-02, -4.9438e-01,
          -9.8271e-01],
         [-6.7902e-02,  1.4848e-01, -1.4579e-01,  3.4029e-01,  3.2356e-01,
           1.5623e-02,  7.5027e-02,  3.0407e-01,  6.9851e-01, -2.5473e-02,
          -9.7975e-02, -1.2705e-01, -5.2653e-01, -2.3293e-01, -6.0229e-01,
           1.0009e-01],
         [ 1.2905e-01,  9.6986e-02, -1.3567e-01,  1.0981e-01,  4.7026e-01,
           9.7366e-02, -5.9666e-02,  1.1595e-02,  3.5564e-01, -5.9338e-02,
          -2.2563e-01,  1.6187e-02, -3.2211e-01, -8.1299e-02, -3.3274e-01,
          -1.7639e-01],
         [ 1.0654e-01,  5.9007e-02, -1.1789e-01, -3.8482e-02,  4.6245e-01,
           3.5215e-02,  8.5292e-02,  2.9452e-01,  2.9620e-01, -5.8692e-02,
          -3.9740e-01,  8.1170e-02, -3.4832e-01,  8.9159e-02, -2

In [ ]:
import torch

# Setup simple dimensions
B = 2          # Batch size
T = 4          # Sequence length
head_size = 64 # The communication channel size (d_k)

# 1. Create random Queries and Keys
# torch.randn automatically creates tensors with a variance of ~1.0
q = torch.randn(B, T, head_size)
k = torch.randn(B, T, head_size)

print("--- BEFORE MULTIPLICATION ---")
print(f"Variance of q: {q.var().item():.4f}")
print(f"Variance of k: {k.var().item():.4f}")

# 2. The Unscaled Dot Product (The Explosion)
# We multiply them without the root d_k scaling
wei_unscaled = q @ k.transpose(-2, -1)

print("\n--- AFTER MULTIPLICATION (UNSCALED) ---")
print(f"Variance of wei: {wei_unscaled.var().item():.4f}")
print(f"Notice it is roughly equal to our head_size of {head_size}!")

# 3. The Fix (Scaled)
wei_scaled = wei_unscaled / (head_size ** 0.5)

print("\n--- AFTER SCALING FIX ---")
print(f"Variance of scaled wei: {wei_scaled.var().item():.4f}")

--- BEFORE MULTIPLICATION ---
Variance of q: 1.1972
Variance of k: 0.9534

--- AFTER MULTIPLICATION (UNSCALED) ---
Variance of wei: 63.6146
Notice it is roughly equal to our head_size of 64!

--- AFTER SCALING FIX ---
Variance of scaled wei: 0.9940
